## ELT: Registro de Atendimentos da Defesa Civil do Recife

### Importação das bibliotecas

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, inspect

### Conexão com o banco
As credenciais do PostgreSQL são armazenadas no arquivo **.env**, fora do código. Antes da carga, uma conexão de teste confirma que o banco está disponível.

In [2]:
# lê as credenciais do arquivo .env
load_dotenv("../.env")

usuario = os.getenv("DB_USER")
senha = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
porta = os.getenv("DB_PORT")
banco = os.getenv("DB_NAME")

database_url = f"postgresql+psycopg2://{usuario}:{senha}@{host}:{porta}/{banco}"
engine = create_engine(database_url)

# testa a conexão (se falhar, interrompe a execução do código)
with engine.connect():
    print("Conexão criada com sucesso.")

Conexão criada com sucesso.


### Extração dos dados

In [3]:
dfs_brutos = {
    "raw_atendimentos_2022": pd.read_csv("data/atendimentos-2022.csv", sep=";", dtype=str, encoding="utf-8"),
    "raw_atendimentos_2023": pd.read_csv("data/atendimentos-2023.csv", sep=";", dtype=str, encoding="utf-8"),
    "raw_atendimentos_2024": pd.read_csv("data/atendimentos-2024.csv", sep=";", dtype=str, encoding="utf-8")
}

for nome_tabela, df_raw in dfs_brutos.items():
    print(f"{nome_tabela}: {df_raw.shape}")

raw_atendimentos_2022: (90425, 16)
raw_atendimentos_2023: (83164, 16)
raw_atendimentos_2024: (80173, 16)


### Carregamento dos dados
As bases são carregadas no PostgreSQL como tabelas brutas, identificadas pelo prefixo *raw*. Depois, a contagem de linhas de cada tabela no banco é comparada com a das bases lidas.

#### Carga das tabelas brutas

In [4]:
for nome_tabela, df_raw in dfs_brutos.items():

    if inspect(engine).has_table(nome_tabela, schema="public"):
        print(f"{nome_tabela} já existe. Mantendo tabela.")
        continue

    df_raw.to_sql(
        nome_tabela,
        con=engine,
        schema="public",
        if_exists="fail",
        index=False
    )

    print(f"{nome_tabela} carregada.")

raw_atendimentos_2022 já existe. Mantendo tabela.
raw_atendimentos_2023 já existe. Mantendo tabela.
raw_atendimentos_2024 já existe. Mantendo tabela.


#### Conferência no banco

In [5]:
for nome_tabela in dfs_brutos:
    total = pd.read_sql(
        f"SELECT COUNT(*) AS total FROM {nome_tabela}",
        con=engine
    )["total"][0] # extrai o valor da contagem que vem num dataframe 1x1

    print(f"{nome_tabela}: {total} linhas no banco.")

raw_atendimentos_2022: 90425 linhas no banco.
raw_atendimentos_2023: 83164 linhas no banco.
raw_atendimentos_2024: 80173 linhas no banco.


### Transformação dos dados
No ELT, as transformações acontecem dentro do banco, nos modelos SQL do projeto dbt **transformacao_atendimentos**. Antes de executar as células abaixo, rode `dbt build` na pasta `transformacao_atendimentos/`, conforme o README: elas consultam tabelas criadas pelo dbt.

O primeiro modelo é a *staging*, que segue os mesmos passos do ETL: correção de 2022, consolidação das bases e padronização das colunas. A consulta abaixo confere o resultado, contando as linhas por ano de origem.

In [6]:
# confere a staging criada pelo dbt
pd.read_sql(
    "SELECT ano_origem, COUNT(*) AS total FROM elt.stg_atendimentos GROUP BY ano_origem ORDER BY ano_origem",
    con=engine
)

,ano_origem,total
0,2022,90425
1,2023,83164
2,2024,80173


#### Conferência das dimensões tipo de ação e risco
Os modelos **int_atendimentos_acao_risco**, **dim_tipo_acao** e **dim_risco** fazem em SQL a mesma padronização do ETL. A tabela de-para do tipo de ação fica no seed **de_para_tipo_acao**. A consulta abaixo confere o tamanho das dimensões: 41 tipos de ação e 6 graus de risco, contando o "não informado".

In [7]:
# confere as dimensões criadas pelo dbt
for dimensao in ["dim_tipo_acao", "dim_risco"]:
    total = pd.read_sql(f"SELECT COUNT(*) AS total FROM elt.{dimensao}", con=engine)["total"][0]
    print(f"{dimensao}: {total} linhas.")

pd.read_sql("SELECT * FROM elt.dim_risco ORDER BY id_risco_sk", con=engine)

dim_tipo_acao: 41 linhas.
dim_risco: 6 linhas.


,id_risco_sk,codigo_risco,descricao_risco,nivel_risco
0,-1,não informado,não informado,NaN
1,1,S/R,sem risco,0.0
2,2,R1,baixo,1.0
3,3,R2,médio,2.0
4,4,R3,alto,3.0
5,5,R4,muito alto,4.0
